# AgriSmart – Google Colab Model Training

This notebook trains the **same model files expected by the AgriSmart VS Code project ZIP**.

It downloads both datasets directly from their Hugging Face sources, trains:
1. **Crop disease classifier** – PlantVillage, 14 crop species / 38 classes.
2. **Crop yield regressor** – India crop-yield dataset.

At the end, it creates `AgriSmart_Models.zip`. Extract that ZIP into the `models` folder of your VS Code project.

**Colab setup:** Runtime → Change runtime type → T4 GPU (if available). Then run cells from top to bottom.

**Important:** Training can take time. Keep the Colab tab/session active. PlantVillage is a large download (about 2.2 GB).

## 1. Install the required libraries

This notebook uses Colab's existing TensorFlow installation. It does not install or upgrade pandas, PyArrow, TensorFlow, or CUDA packages, avoiding unnecessary environment conflicts.

In [ ]:
!pip -q install "datasets>=2.18,<4" "huggingface_hub>=0.23" scikit-learn==1.6.1 joblib dill==0.4.1

In [ ]:
import os, json, zipfile, shutil, random, math
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

WORK_DIR = Path("/content/AgriSmart_Training")
DATA_DIR = WORK_DIR / "data"
MODEL_DIR = WORK_DIR / "models"
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

## 2. Download PlantVillage directly from Hugging Face

Source: `mohanty/PlantVillage`  
Dataset file: `data.zip`

The downloaded archive contains the PlantVillage leaf images. The next cell locates the directory containing the crop/disease class folders automatically.

In [ ]:
from huggingface_hub import hf_hub_download

plant_zip = hf_hub_download(
    repo_id="mohanty/PlantVillage",
    filename="data.zip",
    repo_type="dataset"
)
print("Downloaded PlantVillage archive:", plant_zip)
print("Archive size (GB):", round(os.path.getsize(plant_zip) / (1024**3), 2))

plant_extract = DATA_DIR / "PlantVillage"
plant_extract.mkdir(parents=True, exist_ok=True)

if not any(plant_extract.iterdir()):
    with zipfile.ZipFile(plant_zip, "r") as zf:
        zf.extractall(plant_extract)

print("Extraction complete.")

## 3. Find image classes and create a stratified train/validation split

The class folder names are retained exactly (for example, `Tomato___Early_blight`). These names are what the Streamlit app uses to display crop and disease predictions.

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

def image_count(folder):
    try:
        return sum(1 for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS)
    except Exception:
        return 0

candidates = []
for folder in [plant_extract] + [p for p in plant_extract.rglob("*") if p.is_dir()]:
    subdirs = [p for p in folder.iterdir() if p.is_dir()]
    valid = [p for p in subdirs if image_count(p) > 0]
    if len(valid) >= 10:
        candidates.append((len(valid), image_count(folder), folder))

if not candidates:
    raise RuntimeError(
        "Could not locate the PlantVillage class folders after extraction. "
        "Inspect the extracted archive structure in the Colab Files panel."
    )

candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
CLASS_ROOT = candidates[0][2]
print("Detected image class folder:", CLASS_ROOT)

class_names = sorted(
    p.name for p in CLASS_ROOT.iterdir()
    if p.is_dir() and image_count(p) > 0
)

file_paths, labels = [], []
for label_id, class_name in enumerate(class_names):
    class_dir = CLASS_ROOT / class_name
    for p in class_dir.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            file_paths.append(str(p))
            labels.append(label_id)

print("Number of classes:", len(class_names))
print("Number of images:", len(file_paths))
print("First class names:", class_names[:8])

if len(class_names) != 38:
    print("WARNING: Expected 38 PlantVillage classes; found", len(class_names),
          ". Check the detected directory before training.")

train_paths, val_paths, train_labels, val_labels = train_test_split(
    file_paths, labels, test_size=0.20, random_state=SEED, stratify=labels
)

print("Training images:", len(train_paths))
print("Validation images:", len(val_paths))

## 4. Build TensorFlow input pipelines

Images are resized to 224×224 RGB and scaled to 0–1, matching `backend/disease_detection.py` in the VS Code ZIP.

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE
NUM_CLASSES = len(class_names)

def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, tf.cast(label, tf.int32)

train_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
train_ds = train_ds.shuffle(min(len(train_paths), 10000), seed=SEED)
train_ds = train_ds.map(load_image, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
val_ds = val_ds.map(load_image, num_parallel_calls=AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

for xb, yb in train_ds.take(1):
    print("Image batch:", xb.shape, "Label batch:", yb.shape)

## 5. Train the crop disease detection model

MobileNetV2 is used as a pretrained feature extractor, followed by a 38-class classification head. A short fine-tuning stage then adapts the upper layers to leaf images.

In [ ]:
base_model = keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

inputs = keras.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.25)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
disease_model = keras.Model(inputs, outputs)

disease_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3,
                                  restore_best_weights=True),
    keras.callbacks.ModelCheckpoint(
        str(MODEL_DIR / "disease_best.keras"),
        monitor="val_accuracy", save_best_only=True
    )
]

history_head = disease_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    callbacks=callbacks
)

In [ ]:
# Fine-tune only the last part of MobileNetV2
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

disease_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_fine = disease_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=2,
                                      restore_best_weights=True)
    ]
)

val_loss, val_accuracy = disease_model.evaluate(val_ds, verbose=1)
print(f"Validation accuracy: {val_accuracy:.4f}")

## 6. Save disease model files for the VS Code app

These exact names are required by `backend/disease_detection.py`:
- `disease_model.keras`
- `class_names.json`

In [ ]:
disease_model_path = MODEL_DIR / "disease_model.keras"
disease_model.save(disease_model_path)

with open(MODEL_DIR / "class_names.json", "w", encoding="utf-8") as f:
    json.dump(class_names, f, ensure_ascii=False, indent=2)

print("Saved:", disease_model_path)
print("Saved:", MODEL_DIR / "class_names.json")

## 7. Download the Indian crop-yield dataset directly from Hugging Face

Source: `dhyann2815/india-crop-yield-prediction`

The dataset contains state, crop, season, year, area, rainfall, fertilizer, pesticide, and yield columns. Its dataset card notes that recent-year records include trend-based extensions, so treat model estimates as experimental rather than official forecasts.

In [ ]:
from datasets import load_dataset

yield_data = load_dataset("dhyann2815/india-crop-yield-prediction")
print(yield_data)
print("Train columns:", yield_data["train"].column_names)

yield_train_df = yield_data["train"].to_pandas()
yield_test_df = yield_data["test"].to_pandas()

display(yield_train_df.head())
print("Train rows:", len(yield_train_df), "Test rows:", len(yield_test_df))

## 8. Train the crop-yield prediction model

The VS Code website asks for fertilizer and pesticide use in **kg/ha**. The source dataset stores these as total quantities, so this cell converts them to per-hectare values by dividing by cultivated area and multiplying by 1,000.

The exported object is a scikit-learn Pipeline. It accepts the same raw text/numeric columns that the Streamlit app supplies.

In [ ]:
required = [
    "Year", "State", "Crop", "Season", "Area",
    "Annual_Rainfall", "Fertilizer", "Pesticide", "Yield"
]
missing = [c for c in required if c not in yield_train_df.columns]
if missing:
    raise ValueError(f"Yield dataset is missing expected columns: {missing}")

def prepare_yield_frame(df):
    df = df[required].copy()
    for col in ["Year", "Area", "Annual_Rainfall", "Fertilizer", "Pesticide", "Yield"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=required)
    df = df[df["Area"] > 0].copy()
    # Source data quantities are total quantities; app inputs are kg per hectare.
    df["Fertilizer"] = (df["Fertilizer"] / df["Area"]) * 1000.0
    df["Pesticide"] = (df["Pesticide"] / df["Area"]) * 1000.0
    return df

yield_train_df = prepare_yield_frame(yield_train_df)
yield_test_df = prepare_yield_frame(yield_test_df)

features = [
    "Year", "State", "Crop", "Season", "Area",
    "Annual_Rainfall", "Fertilizer", "Pesticide"
]
categorical_features = ["State", "Crop", "Season"]
numeric_features = ["Year", "Area", "Annual_Rainfall", "Fertilizer", "Pesticide"]

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numeric", "passthrough", numeric_features)
    ]
)

yield_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=250,
        min_samples_leaf=2,
        random_state=SEED,
        n_jobs=-1
    ))
])

X_train = yield_train_df[features]
y_train = yield_train_df["Yield"]
X_test = yield_test_df[features]
y_test = yield_test_df["Yield"]

yield_pipeline.fit(X_train, y_train)
predictions = yield_pipeline.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f"MAE:  {mae:.4f} tonnes/hectare")
print(f"RMSE: {rmse:.4f} tonnes/hectare")
print(f"R²:   {r2:.4f}")

## 9. Save the yield model in the exact format expected by VS Code

`backend/yield_prediction.py` loads a dictionary with two keys: `model` and `features`.

In [ ]:
yield_bundle = {
    "model": yield_pipeline,
    "features": features
}
yield_model_path = MODEL_DIR / "yield_model.joblib"
joblib.dump(yield_bundle, yield_model_path)
print("Saved:", yield_model_path)

## 10. Quick compatibility test

This reproduces the input structure used by the Streamlit website.

In [ ]:
sample_input = pd.DataFrame([{
    "Year": 2026,
    "State": "Karnataka",
    "Crop": "Rice",
    "Season": "Kharif",
    "Area": 1.0,
    "Annual_Rainfall": 850.0,
    "Fertilizer": 100.0,
    "Pesticide": 1.0
}])[features]

sample_prediction = float(yield_pipeline.predict(sample_input)[0])
print("Sample predicted yield:", max(0.0, sample_prediction), "tonnes/hectare")

## 11. Package all trained files for the VS Code project

The final ZIP contains exactly the three model files the supplied VS Code project expects.

In [ ]:
# Remove temporary checkpoint so only app-required artifacts are packaged
best_checkpoint = MODEL_DIR / "disease_best.keras"
if best_checkpoint.exists():
    best_checkpoint.unlink()

export_zip = Path("/content/AgriSmart_Models.zip")
with zipfile.ZipFile(export_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for filename in ["disease_model.keras", "class_names.json", "yield_model.joblib"]:
        file_path = MODEL_DIR / filename
        if not file_path.exists():
            raise FileNotFoundError(f"Required model artifact missing: {file_path}")
        zf.write(file_path, arcname=filename)

print("Created:", export_zip)
print("ZIP size (MB):", round(export_zip.stat().st_size / (1024**2), 2))

from google.colab import files
files.download(str(export_zip))

## 12. Install the trained models in your VS Code ZIP project

1. Download and extract `AgriSmart_Models.zip` on your computer.
2. Open the extracted `AgriSmart_AI_Project` folder from your VS Code ZIP.
3. Open its `models` folder.
4. Copy these three files into that `models` folder:
   - `disease_model.keras`
   - `class_names.json`
   - `yield_model.joblib`
5. Your project structure should look like this:

```text
AgriSmart_AI_Project/
├── app.py
├── requirements.txt
├── backend/
└── models/
    ├── disease_model.keras
    ├── class_names.json
    └── yield_model.joblib
```

Then run the website from the VS Code terminal:

```powershell
streamlit run app.py
```

### Dataset and model limitations

- PlantVillage images were collected in controlled conditions. Real farm photos may produce lower accuracy; test with varied field images before claiming field-level reliability.
- A validation score is not proof of real-world performance. Use a separate field-image test set where possible.
- The yield dataset is community-published and includes extrapolated recent-year values according to its dataset card. Verify against official Indian agricultural statistics before using results for decisions.
- Predictions and recommendations are educational estimates, not agronomic prescriptions.